# 09 — repertoire & naturalness (OAS 직접 다운로드)

> 본문: [`09_repertoire.md`](09_repertoire.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 8초** (실측)

**이 노트북은 도구를 직접 돌립니다.** **진짜 OAS data unit** 을 직접 내려받아 CDR3 길이 분포를 만들고, 후보 항체의 위치를 재요.
각 절은 **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서예요.
내가 만든 결과는 `my_run/` 에 쌓이고, 저장소에 커밋된 `data/` 는 **대조군(레퍼런스)** 으로만 씁니다.
어느 단계를 건너뛰거나 실패해도 `resolve()` 가 `data/` 로 폴백해서 뒤 절이 계속 돌아가요.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

**Colab**: 이 셀을 그대로 실행하면 클론 → 챕터 폴더 이동 → 필요한 도구 설치까지 한 번에 끝나요.
**로컬**: 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "09_repertoire"
PIP_PKGS = "pandas matplotlib anarci abnumber"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True        # ANARCI 계열은 HMMER 의 hmmscan 실행파일이 필요해요 (pip 로는 안 깔려요)
PIN_TRANSFORMERS = None    # IgFold 체크포인트가 요구하는 transformers 버전(없으면 None)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

if NEED_HMMER and shutil.which("hmmscan") is None:
    # ANARCI 는 내부적으로 hmmscan 을 호출해요. pip install anarci 만으로는 안 깔려요.
    if IN_COLAB:
        _run("apt-get -qq update")                       # 인덱스가 낡으면 install 이 404 로 죽는다
        _run("apt-get -qq install -y hmmer")             # ← ANARCI 가 부르는 hmmscan
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if "igfold" in PIP_PKGS and importlib.util.find_spec("pkg_resources") is None:
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 해요.
    _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
    importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


if PIN_TRANSFORMERS:
    # IgFold 체크포인트에는 옛 transformers 의 토크나이저 객체가 pickle 돼 있어요.
    # 최신 transformers(5.x) 로는 unpickle 이 실패해서(Trie/BasicTokenizer 없음) 버전을 맞춥니다.
    _ver = subprocess.run([sys.executable, "-c",
                           "import transformers;print(transformers.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    if not _ver.startswith("4."):
        print(f"[transformers {_ver or 'none'} → {PIN_TRANSFORMERS}] IgFold 체크포인트 호환 버전으로 맞춥니다")
        _run(f'"{sys.executable}" -m pip -q install "transformers=={PIN_TRANSFORMERS}"')

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — OAS data unit 다운로드 + CDR3 길이 집계 (본문 9.1)

```bash
python scripts/fetch_oas_unit.py --out my_run/oas_subset.tsv.gz
python scripts/oas_cdr3_length.py my_run/oas_subset.tsv.gz --column cdr3_aa \
    --out my_run/oas_cdr3_length_summary.csv
```
받는 unit: **Eliyahu et al. 2018 · human PBMC · heavy IgM · run ERR2843400** (productive 17,807 서열).
OAS 원본 파일은 **첫 줄이 메타데이터(JSON)** 라 그냥 읽으면 컬럼을 못 찾아요 — 스크립트가 자동 처리합니다.

In [ ]:
ok_dl = run_tool([PY, SCRIPTS/"fetch_oas_unit.py", "--out", "my_run/oas_subset.tsv.gz"])
ok_sum = run_tool([PY, SCRIPTS/"oas_cdr3_length.py", resolve("oas_subset.tsv.gz"),
                   "--column", "cdr3_aa", "--out", "my_run/oas_cdr3_length_summary.csv"])

## 2) 내 결과 확인 — 분포 통계 + 후보 항체의 위치 (본문 9.2)

In [ ]:
import pandas as pd
from abnumber import Chain

# 후보(demo) 항체의 CDR-H3 길이도 직접 계산해요 (IMGT 정의 = OAS cdr3_aa 와 같은 기준)
seqs, name = {}, None
for line in open("data/demo_mab.fa"):
    line = line.strip()
    if line.startswith(">"): name = line[1:].split()[0]; seqs[name] = ""
    elif name: seqs[name] += line
heavy = Chain(list(seqs.values())[0], scheme="imgt")
cand = len(heavy.cdr3_seq)
print(f"후보 항체 CDR-H3 (IMGT) = {heavy.cdr3_seq} → {cand} aa")

s = pd.read_csv(resolve("oas_cdr3_length_summary.csv"))
n = s["count"].sum()
mean = (s["cdr3_len"] * s["count"]).sum() / n
pct = 100 * s.loc[s["cdr3_len"] <= cand, "count"].sum() / n
print(f"\nOAS: n={n:,} 서열 | 평균 CDR3 {mean:.1f} aa | 범위 {s.cdr3_len.min()}–{s.cdr3_len.max()} aa")
print(f"후보({cand} aa) 는 분포의 하위 {pct:.0f} percentile → 자연 human 분포의 한복판")

## 3) 내 결과 확인 — CDR3 분포 그래프 (본문 9.2)

In [ ]:
from antibody_viz import cdr3_length_distribution
from IPython.display import Image
png = "my_run/09_cdr3_length.png"
cdr3_length_distribution(resolve("oas_cdr3_length_summary.csv"),
    "OAS (Eliyahu 2018, human IgM heavy) — CDR3 length distribution",
    png, highlight_len=cand, highlight_label="demo CDR-H3")
Image(png)

## 4) 레퍼런스 대조 — 커밋된 OAS 서브셋과 같은가

`data/oas_subset.tsv.gz` 는 **같은 OAS data unit 을 2026-07-14 에 받아 커밋해 둔 실제 데이터**예요
(합성 데이터 아님). 내가 방금 받은 것과 집계가 같아야 정상입니다.

In [ ]:
import pandas as pd, pathlib
ref = pd.read_csv("data/oas_cdr3_length_summary.csv")
if pathlib.Path("my_run/oas_cdr3_length_summary.csv").exists():
    mine = pd.read_csv("my_run/oas_cdr3_length_summary.csv")
    same = mine.equals(ref)
    print(f"내 집계 n={mine['count'].sum():,} / 레퍼런스 n={ref['count'].sum():,} | 완전 일치 = {same}")
    if not same:
        m = mine.merge(ref, on="cdr3_len", how="outer", suffixes=("_mine","_ref")).fillna(0)
        print(m[m.count_mine != m.count_ref].head(10).to_string(index=False))
        print("(OAS 가 data unit 을 갱신하면 차이가 날 수 있어요)")
else:
    print("my_run 집계가 없어 대조를 건너뜁니다.")

## 5) 보너스 — 내가 받은 unit 의 V/J germline usage

In [ ]:
import pandas as pd
rep = pd.read_csv(resolve("oas_subset.tsv.gz"), sep="\t")
for col, label in [("v_call", "IGHV"), ("j_call", "IGHJ")]:
    top = rep[col].astype(str).str.split("*").str[0].value_counts().head(5)
    print(f"[{label} top5]")
    for g, n in top.items():
        print(f"   {g:12s} {n:5d}  ({100*n/len(rep):.1f}%)")
print("\n[심화] 후보 항체의 V gene 이 이 목록에 없다면 '희귀 germline' — 왜 드문지 설명할 수 있어야 해요(9.3).")

> 다음 → 본문 [10. 부록](../10_appendix/10_appendix.md)